# WEEK 2: Model Training & Intelligence Layers
## Google Colab Notebook for Fine-tuning CBT Chatbot

**Goal:** Fine-tune DialoGPT model with LoRA adapters and build intelligence layers.

This notebook is designed to run in Google Colab (GPU T4 recommended).

### Prerequisites:
1. Upload tokenized dataset to Google Drive: `data/processed/tokenized/`
2. Have HuggingFace account and token ready (optional, but recommended)

### Outline:
- Step 1: Mount Google Drive
- Step 2: Install dependencies
- Step 3: Load tokenized dataset
- Step 4: Load base model (DialoGPT-medium)
- Step 5: Apply LoRA adapters
- Step 6: Configure training arguments
- Step 7: Fine-tune and save checkpoint

In [ ]:
# Step 1: Mount Google Drive to persist checkpoints
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

In [ ]:
# Step 2: Install transformers, datasets, accelerate, peft in Colab environment
import subprocess
import sys

packages = [
    'transformers[torch]==4.35.2',
    'datasets==2.14.5',
    'accelerate==0.24.1',
    'peft==0.7.1',
    'torch==2.1.1',
    'tensorboard'
]

print("Installing required packages...")
for pkg in packages:
    print(f"  Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✓ All packages installed")

# Verify GPU availability
import torch
print(f"\n✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Load tokenized dataset from Drive
from datasets import DatasetDict
import os

drive_path = '/content/drive/MyDrive/codemind/data/processed/tokenized'
print(f"Loading dataset from {drive_path}...")

# Check if path exists
if not os.path.exists(drive_path):
    print(f"⚠ Path not found: {drive_path}")
    print("Make sure you've uploaded the tokenized dataset to Google Drive")
    print("Expected structure: My Drive/codemind/data/processed/tokenized/")
else:
    # Load dataset
    dataset = DatasetDict.load_from_disk(drive_path)
    print(f"✓ Dataset loaded!")
    print(f"  Train samples: {len(dataset['train'])}")
    print(f"  Validation samples: {len(dataset['validation'])}")
    print(f"  Test samples: {len(dataset['test'])}")
    
    # Show sample
    print(f"\nSample train batch (input_ids length): {len(dataset['train'][0]['input_ids'])}")

In [ ]:
# Step 4: Load microsoft/DialoGPT-medium from HuggingFace
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "microsoft/DialoGPT-medium"
print(f"Loading model: {model_name}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
print(f"✓ Tokenizer loaded (vocab size: {len(tokenizer)})")

# Load model
model = AutoModelForCausalLM.from_pretrained(model_name)
print(f"✓ Model loaded")
print(f"  Parameters: {model.num_parameters():,}")
print(f"  Device: {next(model.parameters()).device}")

In [ ]:
# Step 5: Apply LoRA adapters using peft for memory-efficient fine-tuning
from peft import get_peft_model, LoraConfig, TaskType

print("Configuring LoRA adapters...")

lora_config = LoraConfig(
    r=8,  # LoRA rank
    lora_alpha=16,  # LoRA scaling
    target_modules=["c_attn"],  # Target attention weights in GPT-2
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
print("✓ LoRA adapters applied")

# Show trainable parameters
model.print_trainable_parameters()
total_params = model.num_parameters()
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Reduction: {(1 - trainable_params/total_params)*100:.1f}%")

In [ ]:
# Step 6: Define TrainingArguments — batch_size=4, epochs=3, lr=5e-5
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Training output directory
output_dir = '/content/drive/MyDrive/codemind/models/fine_tuned'
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    fp16=torch.cuda.is_available(),  # Use fp16 if GPU available
    report_to=["tensorboard"],
    push_to_hub=False,
    seed=42
)

print("✓ Training arguments configured:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Output dir: {output_dir}")

In [ ]:
# Step 7: Train using Trainer API and save best checkpoint to Drive
print("\n" + "="*60)
print("WEEK 2: FINE-TUNING DIALOGPT WITH LoRA")
print("="*60)

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    data_collator=data_collator,
)

# Start training
print("\n[Status] Starting fine-tuning...\n")
train_result = trainer.train()

# Save final model
print("\nSaving final checkpoint...")
final_checkpoint = os.path.join(output_dir, 'checkpoint-final')
trainer.save_model(final_checkpoint)
tokenizer.save_pretrained(final_checkpoint)

print(f"✓ Model saved to: {final_checkpoint}")
print(f"\n" + "="*60)
print("WEEK 2 TRAINING COMPLETE!")
print("="*60)
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"\nNext steps:")
print(f"1. Download checkpoint from: {output_dir}")
print(f"2. Place in: models/fine_tuned/")
print(f"3. Build intelligence layers (sentiment, distortion, crisis)")
print("="*60)